# Ejercicio 2: Transcripción del ADN a ARN y Extensión con Biopython

## 📖 Introducción y Fundamento Teórico Resumido

> **Objetivo:** Comprender la transcripción génica, discriminar inequívocamente entre cadena molde y codificante, delimitar el promotor frente a la región codificante y evaluar bioinformáticamente el impacto de la polaridad.

### Resumen de los puntos teóricos del enunciado:

1. **Secuencia de partida:**
   $$\begin{aligned}
   \text{Hebra superior:} \quad & 5'\text{ – ATG CCT GAA TGC – }3' \\
   \text{Hebra inferior:} \quad & 3'\text{ – TAC GGA CTT ACG – }5'
   \end{aligned}$$

2. **Identificación de la cadena molde:**
   - La **ARN polimerasa** sintetiza exclusivamente en dirección **$5' \to 3'$**. En consecuencia, debe desplazarse sobre una hebra molde con orientación antiparalela, es decir, **$3' \to 5'$**.
   - En el código genético, la traducción inicia con el codón canónico **$5'\text{-AUG-}3'$**. Para generar este codón, el molde debe contener el triplete complementario **$3'\text{-TAC-}5'$**.
   - **Cadena molde (*antisentido*):** Es la **hebra inferior** ($3'\text{ – TAC GGA CTT ACG – }5'$).
   - **Cadena codificante (*sentido*):** Es la **hebra superior** ($5'\text{ – ATG CCT GAA TGC – }3'$). Su secuencia coincide con el ARNm resultante (salvo que tiene $T$ en vez de $U$).

3. **Transcrito de ARNm correspondiente:**
   Apareando con la hebra molde ($A \to U$, $T \to A$, $C \to G$, $G \to C$):
   $$\mathbf{5'\text{ – AUG CCU GAA UGC – }3'}$$
   - **Codón 1:** $5'\text{-AUG-}3' \implies$ Metionina (Met / M) — *Codón de Inicio*
   - **Codón 2:** $5'\text{-CCU-}3' \implies$ Prolina (Pro / P)
   - **Codón 3:** $5'\text{-GAA-}3' \implies$ Ácido Glutámico (Glu / E)
   - **Codón 4:** $5'\text{-UGC-}3' \implies$ Cisteína (Cys / C)

4. **Región promotora vs. Región codificante:**
   - **Región Promotora:** Secuencia reguladora ubicada *río arriba* (*upstream*, hacia el extremo $5'$ de la hebra codificante). **No se traduce a proteína**. Su función es reclutar la ARN polimerasa (mediante motivos conservados como la caja TATA o las cajas $-35/-10$) y orientar la dirección de la transcripción.
   - **Región Codificante (CDS / ORF):** Comienza en el codón de inicio ($AUG$) y se extiende en tripletes colineales continuos hasta un codón de parada ($UAA, UAG, UGA$). Dicta directamente la secuencia primaria de aminoácidos de la proteína.

# 💻 Extensión con Biopython: Guía Paso a Paso

> **Enunciado de la extensión:**
> *Crea un script que lea un archivo FASTA con ADN y produzca la secuencia de ARNm. Experimenta cambiando la orientación de la hebra y observa qué ocurre.*

A continuación, implementaremos y analizaremos el script de forma guiada e interactiva.

### Paso 1: Importar módulos y localizar el archivo FASTA de entrada

Para este ejercicio utilizaremos:
- `pathlib.Path`: Para gestionar rutas de archivos de manera robusta y multiplataforma.
- `Bio.SeqIO`: El módulo estándar de Biopython para lectura y escritura de formatos biológicos (FASTA, GenBank, FASTQ).
- `Bio.SeqRecord.SeqRecord`: Estructura de datos que asocia una secuencia (`Seq`) con su identificador, nombre y descripción.

Localizaremos el archivo `ejercicio_02_adn.fasta` ubicado en la carpeta `data/` del repositorio.

In [1]:
from pathlib import Path
from Bio import SeqIO
from Bio.SeqRecord import SeqRecord

# Localización robusta del directorio data tanto desde notebooks/ como desde la raíz
base_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
fasta_path = base_dir / "data" / "ejercicio_02_adn.fasta"

print(f"Ruta base del proyecto:   {base_dir}")
print(f"Ruta al archivo FASTA:    {fasta_path}")
print(f"¿Existe el archivo FASTA? {fasta_path.exists()}")

Ruta base del proyecto:   C:\Users\Ayoze\Desktop\Del-ADN-a-la-protenia
Ruta al archivo FASTA:    C:\Users\Ayoze\Desktop\Del-ADN-a-la-protenia\data\ejercicio_02_adn.fasta
¿Existe el archivo FASTA? True


### Paso 2: Lectura del archivo FASTA con `SeqIO.read()`

Un archivo en formato FASTA comienza con una línea de cabecera iniciada con `>` que contiene el identificador y descripción, seguida de las líneas con los nucleótidos.

Cuando sabemos que el archivo contiene **exactamente un único registro**, la función **`SeqIO.read(archivo, "fasta")`** es la vía óptima de carga en Biopython (si el archivo tuviera múltiples registros se usaría `SeqIO.parse`).

In [2]:
# Lectura del registro FASTA
record = SeqIO.read(fasta_path, "fasta")

print("=" * 65)
print("REGISTRO FASTA CARGADO CON BIOPYTHON")
print("=" * 65)
print(f"ID del registro:        {record.id}")
print(f"Descripción completa:   {record.description}")
print(f"Longitud de secuencia:  {len(record.seq)} nucleótidos (pb)")
print(f"Secuencia de ADN:       5'- {record.seq} -3'")
print("=" * 65)

REGISTRO FASTA CARGADO CON BIOPYTHON
ID del registro:        ejercicio_02_adn
Descripción completa:   ejercicio_02_adn Secuencia de ADN dada en el ejercicio 2 (hebra codificante 5'->3')
Longitud de secuencia:  12 nucleótidos (pb)
Secuencia de ADN:       5'- ATGCCTGAATGC -3'


### Paso 3: Transcripción directa de la hebra codificante a ARNm

En las bases de datos biológicas internacionales (NCBI, Ensembl), las secuencias depositadas en formato FASTA representan por estándar la **hebra codificante en orientación $5' \to 3'$**.

Por este motivo, el método **`.transcribe()`** de Biopython está optimizado para tomar la hebra codificante y reemplazar directamente cada Timina ($T$) por Uracilo ($U$), produciendo el transcrito primario de ARNm en sentido $5' \to 3'$.

In [3]:
# Transcripción directa de ADN a ARNm
mrna_direct = record.seq.transcribe()

# Agrupamos en codones (tripletes de nucleótidos)
codones_direct = [str(mrna_direct[i:i+3]) for i in range(0, len(mrna_direct), 3)]

print("=" * 65)
print("1. TRANSCRIPCIÓN DIRECTA (Hebra codificante canónica 5' -> 3')")
print("=" * 65)
print(f"ADN codificante: 5'- {record.seq} -3'")
print(f"ARNm transcrito: 5'- {mrna_direct} -3'")
print(f"Pauta de codones: {codones_direct}")
print("=" * 65)

1. TRANSCRIPCIÓN DIRECTA (Hebra codificante canónica 5' -> 3')
ADN codificante: 5'- ATGCCTGAATGC -3'
ARNm transcrito: 5'- AUGCCUGAAUGC -3'
Pauta de codones: ['AUG', 'CCU', 'GAA', 'UGC']


### Paso 4: Experimento bioinformático — Cambiando la orientación de la hebra

El enunciado solicita: *«Experimenta cambiando la orientación de la hebra y observa qué ocurre»*.

Vamos a evaluar dos situaciones biológicas y bioinformáticas distintas:

1. **Caso A: Transcripción desde la hebra reversa-complementaria (`.reverse_complement()`):**
   Representa la situación biológica en la que un promotor estuviera orientado en sentido opuesto, utilizando como codificante la otra cadena del dúplex.

2. **Caso B: Transcripción tras invertir físicamente el orden (`[::-1]`):**
   Representa la lectura física de la hebra de $3' \to 5'$ tratada incorrectamente como si fuese $5' \to 3'$.

In [4]:
print("=" * 65)
print("2. EXPERIMENTO: ALTERACIÓN DE LA ORIENTACIÓN DE LA HEBRA")
print("=" * 65)

# Caso A: Reversa complementaria (5' -> 3' de la hebra opuesta)
rev_comp_dna = record.seq.reverse_complement()
mrna_rev_comp = rev_comp_dna.transcribe()
codones_rev_comp = [str(mrna_rev_comp[i:i+3]) for i in range(0, len(mrna_rev_comp), 3)]

print("a) Caso Reversa-Complementaria (hebra opuesta en sentido 5' -> 3'):")
print(f"   ADN rev-comp:    5'- {rev_comp_dna} -3'")
print(f"   ARNm resultante: 5'- {mrna_rev_comp} -3'")
print(f"   Codones:         {codones_rev_comp}")

# Caso B: Inversión física de índices (orden 3' -> 5' tratado como 5' -> 3')
reversed_dna = record.seq[::-1]
mrna_reversed = reversed_dna.transcribe()
codones_reversed = [str(mrna_reversed[i:i+3]) for i in range(0, len(mrna_reversed), 3)]

print("\nb) Caso Inversión Física ([::-1]):")
print(f"   ADN invertido:   5'- {reversed_dna} -3'")
print(f"   ARNm resultante: 5'- {mrna_reversed} -3'")
print(f"   Codones:         {codones_reversed}")
print("=" * 65)

2. EXPERIMENTO: ALTERACIÓN DE LA ORIENTACIÓN DE LA HEBRA
a) Caso Reversa-Complementaria (hebra opuesta en sentido 5' -> 3'):
   ADN rev-comp:    5'- GCATTCAGGCAT -3'
   ARNm resultante: 5'- GCAUUCAGGCAU -3'
   Codones:         ['GCA', 'UUC', 'AGG', 'CAU']

b) Caso Inversión Física ([::-1]):
   ADN invertido:   5'- CGTAAGTCCGTA -3'
   ARNm resultante: 5'- CGUAAGUCCGUA -3'
   Codones:         ['CGU', 'AAG', 'UCC', 'GUA']


### Paso 5: Análisis comparativo y conclusiones biológicas del experimento

Comparemos lado a lado los 3 transcritos obtenidos para comprender las repercusiones a nivel del código genético:

In [5]:
print("=" * 75)
print(f"{'Condición':<25} | {'Secuencia ARNm':<16} | {'Tripletes / Codones'}")
print("-" * 75)
print(f"{'1. Codificante canónica':<25} | {str(mrna_direct):<16} | {codones_direct}")
print(f"{'2. Reversa-complementaria':<25} | {str(mrna_rev_comp):<16} | {codones_rev_comp}")
print(f"{'3. Inversión lineal':<25} | {str(mrna_reversed):<16} | {codones_reversed}")
print("=" * 75)

print("\n>> OBSERVACIONES BIOLÓGICAS CLAVE:")
print("1. Polaridad estricta: La ARN polimerasa polimeriza unidireccionalmente en sentido 5' -> 3'.")
print("2. Destrucción del marco de lectura: Solo la orientación canónica contiene el codón")
print("   de inicio 'AUG'. Al invertir la secuencia o tomar la hebra opuesta, el codón de inicio")
print("   desaparece por completo ('GCA' o 'CGU'), impidiendo la traducción normal.")
print("3. Péptido totalmente diferente: Cada triplete codifica para un aminoácido distinto,")
print("   demostrando que la orientación del gen en el cromosoma es crítica y específica.")

Condición                 | Secuencia ARNm   | Tripletes / Codones
---------------------------------------------------------------------------
1. Codificante canónica   | AUGCCUGAAUGC     | ['AUG', 'CCU', 'GAA', 'UGC']
2. Reversa-complementaria | GCAUUCAGGCAU     | ['GCA', 'UUC', 'AGG', 'CAU']
3. Inversión lineal       | CGUAAGUCCGUA     | ['CGU', 'AAG', 'UCC', 'GUA']

>> OBSERVACIONES BIOLÓGICAS CLAVE:
1. Polaridad estricta: La ARN polimerasa polimeriza unidireccionalmente en sentido 5' -> 3'.
2. Destrucción del marco de lectura: Solo la orientación canónica contiene el codón
   de inicio 'AUG'. Al invertir la secuencia o tomar la hebra opuesta, el codón de inicio
   desaparece por completo ('GCA' o 'CGU'), impidiendo la traducción normal.
3. Péptido totalmente diferente: Cada triplete codifica para un aminoácido distinto,
   demostrando que la orientación del gen en el cromosoma es crítica y específica.


### Paso 6: Exportación del ARNm resultante a un nuevo archivo FASTA

En bioinformática, los resultados procesados se almacenan en formatos estandarizados para que sirvan de entrada a los siguientes pasos del pipeline (como la traducción en el Ejercicio 3).

Empaquetamos la secuencia `mrna_direct` dentro de un objeto **`SeqRecord`** y la guardamos en `data/ejercicio_02_arnm.fasta` con **`SeqIO.write()`**.

In [6]:
output_fasta = base_dir / "data" / "ejercicio_02_arnm.fasta"

# Creamos el registro estructurado de ARNm con sus metadatos
mrna_record = SeqRecord(
    mrna_direct,
    id=f"{record.id}_mRNA",
    description="ARNm transcrito primario (5'->3') generado con Biopython",
)

# Escritura en formato FASTA
SeqIO.write(mrna_record, output_fasta, "fasta")

print(f"[OK] Archivo FASTA generado exitosamente en: {output_fasta.name}")

# Verificamos su contenido volviéndolo a leer del disco
verificacion_record = SeqIO.read(output_fasta, "fasta")
print(f"     ID guardado:         {verificacion_record.id}")
print(f"     Secuencia guardada:  {verificacion_record.seq}")

[OK] Archivo FASTA generado exitosamente en: ejercicio_02_arnm.fasta
     ID guardado:         ejercicio_02_adn_mRNA
     Secuencia guardada:  AUGCCUGAAUGC


### Paso 7 (Extra interactivo): Función modular de transcripción con análisis de codones

Para hacer este flujo totalmente reutilizable con cualquier secuencia o gen futuro, definimos una función modular `transcribir_secuencia_adn(secuencia_adn)` que resume todo el proceso y detecta automáticamente si contiene el codón iniciador `AUG`.

In [7]:
from Bio.Seq import Seq

def transcribir_secuencia_adn(secuencia: str) -> dict:
    """
    Transcribe una secuencia de ADN codificante (5' -> 3') a ARNm,
    desglosa sus codones e informa sobre el codón de inicio.
    """
    seq_dna = Seq(secuencia.strip().upper())
    seq_mrna = seq_dna.transcribe()
    codones = [str(seq_mrna[i:i+3]) for i in range(0, len(seq_mrna), 3)]
    
    tiene_inicio = codones[0] == "AUG" if codones else False
    
    print(f"\n>>> Transcripción de: {seq_dna}")
    print(f"    ARNm:              5'- {seq_mrna} -3'")
    print(f"    Codones:           {codones}")
    print(f"    ¿Inicia con AUG?:  {'SÍ (Pauta canónica iniciadora)' if tiene_inicio else 'NO (Sin codón canónico)'}")
    
    return {"adn": seq_dna, "mrna": seq_mrna, "codones": codones, "valido_inicio": tiene_inicio}

# Probamos con la secuencia del ejercicio:
res1 = transcribir_secuencia_adn("ATGCCTGAATGC")

# Probamos con la secuencia invertida:
res2 = transcribir_secuencia_adn("CGTAAGTCCGTA")


>>> Transcripción de: ATGCCTGAATGC
    ARNm:              5'- AUGCCUGAAUGC -3'
    Codones:           ['AUG', 'CCU', 'GAA', 'UGC']
    ¿Inicia con AUG?:  SÍ (Pauta canónica iniciadora)

>>> Transcripción de: CGTAAGTCCGTA
    ARNm:              5'- CGUAAGUCCGUA -3'
    Codones:           ['CGU', 'AAG', 'UCC', 'GUA']
    ¿Inicia con AUG?:  NO (Sin codón canónico)



## 📌 Resumen de lo aprendido

- **Cadena molde vs codificante:** La ARN polimerasa lee la hebra molde en sentido $3' \to 5'$ para sintetizar el ARNm en sentido $5' \to 3'$. La hebra codificante tiene la misma secuencia que el ARNm (cambiando $T$ por $U$).
- **`Bio.SeqIO.read()`:** Permite cargar archivos FASTA directamente como objetos `SeqRecord` estructurados.
- **`Seq.transcribe()`:** Realiza la transcripción biológica directa a ARNm.
- **Impacto de la orientación:** Cambiar la dirección o elegir la hebra contraria destruye la pauta de codones, alterando radicalmente la proteína sintetizada o impidiendo el inicio de la traducción.
- **Exportación FASTA:** `Bio.SeqIO.write()` permite persistir los transcritos generados para su uso en etapas posteriores del pipeline.